In [1]:
from dotenv import load_dotenv
import os 
from google import genai

load_dotenv()
api_key = os.getenv("API_KEY")
client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)

response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Here are a few programming jokes for you:

1.  There are 10 types of people in the world: those who understand binary, and those who don't.
2.  To understand recursion, you must first understand recursion.
3.  A programmer walks into a bar and orders 1.0000000000000000000000000000001 beers."""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='dnXBaJbdENS1vdIP79Cc4As',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=99,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts

In [2]:
response.text

"Here are a few programming jokes for you:\n\n1.  There are 10 types of people in the world: those who understand binary, and those who don't.\n2.  To understand recursion, you must first understand recursion.\n3.  A programmer walks into a bar and orders 1.0000000000000000000000000000001 beers."

In [3]:

print(response.text)

Here are a few programming jokes for you:

1.  There are 10 types of people in the world: those who understand binary, and those who don't.
2.  To understand recursion, you must first understand recursion.
3.  A programmer walks into a bar and orders 1.0000000000000000000000000000001 beers.


In [4]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

ask_llm("Du är en Göteborgare, ge mig ett skämt som är go")

'Jamen se gött de é att du är här å vill ha ett skämt la! Här kommer en go en:\n\nEn göteborgare möter en kompis på stan och frågar:\n"Är du go, eller?"\n\nKompan svarar:\n"Nejdu, jag är la bara här!"\n\nDen är la go, eller hur? Hahaha! Ha de gött nu!'

In [5]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'```json\n[\n    {\n        "address": "Kungsgatan 34A",\n        "price_sek": 4850000,\n        "city": "Stockholm",\n        "monthly_fee": 3200,\n        "area": 55\n    },\n    {\n        "address": "Linnegatan 12",\n        "price_sek": 3100000,\n        "city": "Göteborg",\n        "monthly_fee": 4100,\n        "area": 70\n    },\n    {\n        "address": "Västra Hamngatan 7",\n        "price_sek": 2750000,\n        "city": "Malmö",\n        "monthly_fee": 3800,\n        "area": 65\n    },\n    {\n        "address": "Storgatan 11B",\n        "price_sek": 1950000,\n        "city": "Umeå",\n        "monthly_fee": 3500,\n        "area": 50\n    },\n    {\n        "address": "Östra Storgatan 45",\n        "price_sek": 3500000,\n        "city": "Jönköping",\n        "monthly_fee": 4500,\n        "area": 80\n    }\n]\n```'

In [6]:
print(response)

```json
[
    {
        "address": "Kungsgatan 34A",
        "price_sek": 4850000,
        "city": "Stockholm",
        "monthly_fee": 3200,
        "area": 55
    },
    {
        "address": "Linnegatan 12",
        "price_sek": 3100000,
        "city": "Göteborg",
        "monthly_fee": 4100,
        "area": 70
    },
    {
        "address": "Västra Hamngatan 7",
        "price_sek": 2750000,
        "city": "Malmö",
        "monthly_fee": 3800,
        "area": 65
    },
    {
        "address": "Storgatan 11B",
        "price_sek": 1950000,
        "city": "Umeå",
        "monthly_fee": 3500,
        "area": 50
    },
    {
        "address": "Östra Storgatan 45",
        "price_sek": 3500000,
        "city": "Jönköping",
        "monthly_fee": 4500,
        "area": 80
    }
]
```


In [7]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments
    

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:

apartments.objects

[Apartment(address='Karlbergsvägen 77', city='Stockholm', price_sek=6500000, monthly_fee=3800, area=65),
 Apartment(address='Linnégatan 24', city='Göteborg', price_sek=4200000, monthly_fee=4500, area=70),
 Apartment(address='Davidshallsgatan 12', city='Malmö', price_sek=3100000, monthly_fee=3200, area=60),
 Apartment(address='Dragarbrunnsgatan 50', city='Uppsala', price_sek=2750000, monthly_fee=2900, area=55),
 Apartment(address='Starrgränd 3', city='Stockholm', price_sek=3900000, monthly_fee=4700, area=75)]

In [ ]:
apartments.objects[1].address, apartments.objects[1].city

('Linnégatan 24', 'Göteborg')

In [ ]:
addresses = [
    [home.address, home.city, home.price_sek, home.monthly_fee]
    for home in apartments.objects
    if   4000000 < home.price_sek < 8000000
]

addresses

[['Karlbergsvägen 77', 'Stockholm', 6500000, 3800],
 ['Linnégatan 24', 'Göteborg', 4200000, 4500]]

In [ ]:
import pandas as pd

filtered_homes = [
    home for home in apartments.objects if 4_000_000 < home.price_sek < 8_000_000
]
# Convert to df
df = pd.DataFrame(
    [home.model_dump(include={"address", "city", "price_sek", "monthly_fee"}) for home in filtered_homes]
)
df

,address,city,price_sek,monthly_fee
0,Karlbergsvägen 77,Stockholm,6500000,3800
1,Linnégatan 24,Göteborg,4200000,4500


In [ ]:
df

,address,city,price_sek,monthly_fee
0,Karlbergsvägen 77,Stockholm,6500000,3800
1,Linnégatan 24,Göteborg,4200000,4500


In [ ]:
import duckdb
import dlt

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    for record in df.to_dict(orient="records"):
        yield record
        
pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination= "duckdb",
    dataset_name="staging",
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 0.13 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\salih\OneDrive\Dokument\Github\ai_engineering_salih_morina\code-alongs\07_pydantic_geminai\apartments.duckdb location to store data
Load package 1757331598.140868 is LOADED and contains no failed jobs
